# Day 6 打卡：自动求导（autograd）

来源：[2.5 自动求导](https://zh.d2l.ai/chapter_preliminaries/autograd.html)

**要点摘录（读书笔记）**：
1. `x.requires_grad=True` 让 PyTorch 开始追踪 x 的一切运算
2. `y.backward()` 自动算出 y 对所有被追踪变量的梯度，存进 `x.grad`
3. 默认每次 backward 会累积梯度，新的一轮要 `x.grad.zero_()` 清零
4. 不需要追踪的代码放进 `with torch.no_grad():`，省内存

In [1]:
# 【1】最简单的例子：y=2·xᵀx 对 x 求导。requires_grad=True 开始追踪，backward() 后 x.grad 就是梯度
import torch

x = torch.arange(4.0)
x

tensor([0., 1., 2., 3.])

In [2]:
# 【2】验证：y=2xᵀx 的梯度是 4x，x=1,2,3 → 梯度应为 4,8,12，完全吻合
x.requires_grad_(True)  # 等价于x=torch.arange(4.0,requires_grad=True)
x.grad  # 默认值是None

In [3]:
# 【3】backward 不会自动清零：再算一次，梯度变成累积的 8,16,24
y = 2 * torch.dot(x, x)
y

tensor(28., grad_fn=<MulBackward0>)

In [4]:
# 【4】正确做法：每轮之前 x.grad.zero_() 清零，重新 backward
y.backward()
x.grad

tensor([ 0.,  4.,  8., 12.])

In [5]:
# 【5】把计算分离出计算图：y.detach() 得到不再追踪梯度的常量
x.grad == 4 * x

tensor([True, True, True, True])

In [6]:
# 【6】no_grad 上下文：纯查询/评估时不需要梯度，用它省内存
# 在默认情况下，PyTorch会累积梯度，我们需要清除之前的值
x.grad.zero_()
y = x.sum()
y.backward()
x.grad

tensor([1., 1., 1., 1.])